# 今日学习笔记：为什么是卷积

## 一、 理论核心：李宏毅 CNN 推导
打破“生物视觉”套路，从全连接网络 (Fully Connected) 增加限制 (Restriction) 推导出卷积：
1. **局部性 (Locality) → 感受野 (Receptive Field)**
   - 提取特征（如鸟嘴）无需整图，只需关注局部区域。砍掉大量不必要的全连接。
2. **平移不变性 (Translation Invariance) → 参数共享 (Parameter Sharing)**
   - 同样的特征可能出现在图像的不同位置。同一个卷积核在整张图上滑动计算，权重共用。

## 二、 验收：参数量悬殊对比
**条件**：输入 224x224x3 图像（展平为 150,528 维），目标提取 64 个特征。
* **全连接 (假设仅 1000 个节点)**：150,528 * 1,000 ≈ **1.5 亿** 个参数。
* **卷积层 (假设 3x3 卷积核)**：3 * 3 * 3(输入通道) * 64(输出通道) = **1,728** 个参数（不计 bias）。
* **结论**：局部性导致单个特征提取器尺寸骤降（从 15 万变 27），平移不变性导致不需要为每个位置重复训练。



# 今日学习笔记：贝叶斯公式与条件概率

## 一、 基石：条件概率 (Conditional Probability)
* **核心思想**：在已知前提 (B) 发生的情况下，事件 (A) 发生的概率。相当于把 B 当作新的“全集”。
* **公式**：P(A|B) = P(A ∩ B) / P(B)

## 二、 核心：贝叶斯公式 (Bayes' Theorem)
* **本质**：根据新观测到的数据 (D)，不断更新对原有假设 (H) 的认知。
* **公式**：P(H|D) = [P(D|H) * P(H)] / P(D)

## 三、 必须记熟的三个核心词汇
1. **先验 (Prior) —— P(H)**
   - **含义**：在没有看到任何数据之前，对假设成立的初始预判（经验之谈）。
2. **似然 (Likelihood) —— P(D|H)**
   - **含义**：假设我们的认知/模型是正确的，那么观察到眼下这组数据的可能性有多大。
3. **后验 (Posterior) —— P(H|D)**
   - **含义**：看到了实际数据后，更新后的最终判断（我们在机器学习中真正想要优化的目标）。

## 四、 机器学习终极口诀
在实际训练中，分母 P(D)（证据）通常作为归一化常数被忽略，必须死记的核心关系为：
**后验 ∝ 似然 × 先验**
* **大白话**：最终的判断（后验） = 眼前的证据（似然） 结合 最初的经验（先验）。

# d2l 6.1-6.2 核心复习笔记：从全连接到图像卷积

## 一、 理论演进：全连接是如何变成卷积的？(6.1)

把传统的全连接层（Dense/MLP）直接用来处理图像存在两个致命缺陷：
1. **丧失空间结构**：需要把 2D 图片展平为 1D 向量，丢弃了像素间相邻的拓扑信息。
2. **参数量灾难**：参数量等于 `输入像素数 × 隐藏层节点数`，单层极易飙升至数亿级别，导致显存爆炸且极易过拟合。

**卷积层的本质：对全连接层施加了两个强力的“物理先验限制”**

*   **限制一：平移不变性 (Translation Invariance)**
    *   *物理直觉*：无论特征（如猫耳）出现在图片的左上角还是右下角，模型都该用同样的方法识别它。
    *   *数学操作*：打破全连接中“每个位置拥有独立权重”的设定，强制要求同一个特征提取器在图像各个位置**共享同一套权重（Parameter Sharing）**。
*   **限制二：局部性 (Locality)**
    *   *物理直觉*：识别一个局部特征不需要看整张图，只需关注该特征所在的局部小窗口。
    *   *数学操作*：砍掉全连接中 99% 的连线，强制神经元只和图像上**极小的一个局部区域**相连。
*   **结论：全连接层 + 平移不变性 + 局部性 = 卷积层**

---

## 二、 计算机制：二维互相关运算 (6.2)

深度学习框架（PyTorch/TensorFlow）底层执行的“卷积”，数学严格意义上叫**互相关 (Cross-Correlation)**。

### 1. 运算规则与尺寸公式
*   **计算方式**：将卷积核放到输入矩阵上，**滑动窗口，对应位置逐元素相乘并求和**。
*   **不翻转的原因**：真正的数学卷积需要把核翻转 180 度。但在深度学习中，核的参数是学出来的，学出一个翻转的核和没翻转的核难度完全一样，为了计算效率直接省去翻转步骤。
*   **输出尺寸公式（无填充、步长为 1 时）**：
    *   输出高度 = `H - k_h + 1`
    *   输出宽度 = `W - k_w + 1`
    *   *(注：每次滑动，卷积核都会“吃掉”图片边缘的一圈像素)*

### 2. 物理直觉：边缘检测
卷积核本质上是在计算局部像素的**离散差分（一阶梯度）**。
*   例如，使用 `[1, -1]` 的卷积核横向滑动，当遇到同色区域时，相减为 0（无响应）；当遇到颜色突变的交界处时，相减会产生极大/极小的非零值（边缘高亮）。

---

## 三、 深度学习范式的跨越 与 核心名词

### 1. 从手工设计到网络自主学习
*   **传统 CV**：工程师凭借数学直觉**人工设计**卷积核（如 Sobel 算子）。
*   **深度学习**：随机初始化卷积核，给它输入和标签，通过 **Loss + 反向传播** 让网络自己**学出**最优的矩阵参数。这使得网络能提取人类无法手工设计的抽象语义特征。

### 2. 核心专业名词
*   **特征图 (Feature Map)**：卷积层的输出矩阵。它不再是单纯的图像，而是输入图像在特定卷积核作用下的“响应强度分布图”。
*   **感受野 (Receptive Field)**：特征图上的**某一个点**，在**原始输入图像**上所覆盖的有效观察区域大小。
    *   *进阶理解*：通过堆叠多个小尺寸（如 $3 \times 3$）的卷积层，即使不增加单个卷积核的尺寸，深层神经元的实际感受野也会越来越大，从而能够捕捉更宏观的全局特征。

In [1]:
# 实现 corr2d 函数
import torch 
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

In [2]:
# 包装成 Conv2D
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

In [3]:
X = torch.ones((6, 8))
X[:, 2:6] = 0

K = torch.tensor([[1.0, -1.0]])

Y = corr2d(X, K)

corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [4]:
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')

epoch 2, loss 10.470
epoch 4, loss 2.763
epoch 6, loss 0.875
epoch 8, loss 0.316
epoch 10, loss 0.122


In [5]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9516, -1.0223]])